In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive
from datasets import Dataset
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score


DRIVE_PATH = '/content/drive/MyDrive/TFG_Posdata'
MODEL_NAME = "xlm-roberta-base"
OUTPUT_DIR = os.path.join(DRIVE_PATH, 'models','trained_model_v1')
DATASET_PATH = os.path.join(DRIVE_PATH, 'datasets','bilingual_spam_sms_dataset_enriched.csv')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

def compute_metrics(p):
    pred, labels  = p
    pred = np.argmax(pred, axis=1)

    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    f1 = f1_score(y_true=labels, y_pred=pred, average='weighted')

    return {"accuracy": accuracy, "f1": f1}

drive.mount('/content/drive')

print("Loading dataset...")
df = pd.read_csv(DATASET_PATH)

df_en = df[['text_en', 'label']].rename(columns={'text_en': 'text'})
df_es = df[['text_es', 'label']].rename(columns={'text_es': 'text'})
df_final = pd.concat([df_en, df_es], ignore_index=True)
df_final['label'] = df_final['label'].map({'ham': 0, 'spam': 1})

train_df, test_df = train_test_split(df_final, test_size=0.2, stratify=df_final['label'], random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"Downloading tokenizer {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Datasets are ready. Downloading model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training... (This may take a while)")
trainer.train()

print("Training complete. Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model and tokenizer saved in {OUTPUT_DIR}.")

In [ ]:
from transformers import pipeline

MODEL_PATH = "/content/drive/My Drive/TFG_Posdata/models/trained_model_v1"

print("Cargando el modelo para pruebas...")
classifier = pipeline("text-classification", model=MODEL_PATH, tokenizer=MODEL_PATH)

test_messages = [
    "Hey mom, are you coming for dinner tonight?",  # Ham Inglés
    "URGENTE: Su cuenta ha sido bloqueada. Verifique aqui: http://bit.ly/scam", # Spam Español
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.", # Spam Inglés clásico
    "Hola Juan, al final la reunión se mueve al martes.", # Ham Español
    "Banco Santander: Se ha realizado un cargo de 500€. Si no ha sido usted, cancele aquí: http://fakesite.com", # Spam Español (Phishing)
    "Your package is waiting for delivery. Please confirm your address.", # Spam sin URL (Smishing)
    "152896 es tu código Ticketmaster", #Ham Español
    "La entrega se ha suspendido porque su pedido no tiene numero de casa. Actualice lo antes posible. https://seurn3.top/es" #Spam Español
]

# 3. Probarlos
print("\n--- RESULTADOS DE LA PRUEBA ---")
print(f"{'MENSAJE':<60} | {'PREDICCIÓN':<10} | {'CONFIANZA'}")
print("-" * 90)

for msg in test_messages:
    result = classifier(msg)[0]

    # Traducir etiqueta (LABEL_1 suele ser Spam, LABEL_0 Ham, depende de tu map anterior)
    # En el entrenamiento definimos: {'spam': 1, 'ham': 0}
    label = "🔴 SPAM" if result['label'] == 'LABEL_1' else "🟢 HAM"
    score = result['score'] * 100 # Convertir a porcentaje

    # Recortar mensaje si es muy largo para que quepa en la tabla
    msg_short = (msg[:57] + '...') if len(msg) > 57 else msg

    print(f"{msg_short:<60} | {label:<10} | {score:.2f}%")

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
from transformers import AutoTokenizer
import os
from google.colab import drive

MODEL_PATH = "/content/drive/My Drive/TFG_Posdata/models/trained_model_v1"
ONNX_PATH = "/content/light_model"

drive.mount('/content/drive')

print("Transforming to ONNX (This may take some time...)")
model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    export=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model.save_pretrained(ONNX_PATH)
tokenizer.save_pretrained(ONNX_PATH)

print("Compressing model...")
quantizer = ORTQuantizer.from_pretrained(model)
dqconfig = AutoQuantizationConfig.avx512(is_static=False, per_channel=False)

quantizer.quantize(
    save_dir=ONNX_PATH + "_quantized",
    quantization_config=dqconfig
)

print("Your model is ready!")
